In [10]:
%reload_ext autoreload
%autoreload 2

import os
import pickle
import numpy as np
import json

from fpp.utils.validation import pp_finite_sample_band
from fpp.utils.posterior import multi_corner

import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc_file("../src/fpp/utils/matplotlibrc")

# Coverage

In [11]:
run_name = 'calibration/hmctd10-old-delta-2'
z = pickle.load(open(os.environ['MYSTORE'] + f'/fermi/fermi-prob-prog/outputs/production/fits/{run_name}/p_nominal_actual_dict.p', 'rb'))
print(' '.join(z.keys()))

S_bub S_gce S_ics S_iso S_pib S_psc Sps_dsk n1_dsk n2_dsk n3_dsk sb1_dsk lambdas_dsk Sps_gce n1_gce n2_gce n3_gce sb1_gce lambdas_gce f_bulge_poiss f_bulge_ps gamma_poiss gamma_ps C zs


In [ ]:
if 'pois' in run_name:
    labels = [
        'S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_gce',
        'f_bulge_poiss', 'gamma_poiss',
    ]
else:
    labels = [
        'S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_gce',
        'Sps_dsk', 'Sps_gce',
        'f_bulge_poiss', 'f_bulge_ps', 'gamma_poiss', 'gamma_ps', 'C', 'zs'
    ]

probs = [z[k] for k in labels]
ls_s = ['-'] * 10 + [':'] * 10

fig, ax = plt.subplots()

ax.fill_between([0,1], [0,1], color='lightgray')
if labels is None:
    labels = [None for _ in probs]
for prob, label, ls in zip(probs, labels, ls_s):
    ax.plot(prob[0], prob[1], label=label, ls=ls)

n_run = len(probs[0][0])
lower, upper = pp_finite_sample_band(n_run)
ax.plot(upper, np.linspace(0, 1, n_run), 'k:', label=f'{n_run} sample 95\% \ncontainment')
ax.plot(lower, np.linspace(0, 1, n_run), 'k:')

ax.set(aspect=1, xlim=(0, 1), ylim=(0, 1))
ax.set(xlabel='Nominal coverage', ylabel='Actual coverage', title=run_name)

fig.legend(bbox_to_anchor=(1, 1), loc='upper left', bbox_transform=ax.transAxes)
plt.tight_layout()

# Corner

In [ ]:
point_est = True
truth_dict = json.load(open("../outputs/truths/truth_dict_base230927.json", 'r'))
labels = [
    'S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_gce', 'Sps_dsk', 'Sps_gce',
    'f_bulge_poiss', 'f_bulge_ps', 'gamma_poiss', 'gamma_ps', 'C', 'zs'
]
config_dict = {
    'hmc' : ('/n/home07/yitians/fermi/fermi-prob-prog/outputs/production/fits/calibration/hmc-old-delta-2/4.p', 'k'),
    'pt' : ('/n/home07/yitians/fermi/fermi-prob-prog/outputs/production/fits/calibration/pthmc-old-delta-2/4.p', 'r'),
}

s_in = {}
labels_dict = {}
colors_dict = {}
for key, (path, color) in config_dict.items():
    s = pickle.load(open(path, 'rb'))
    s_in[key] = {k: s[k] for k in labels}
    labels_dict[key] = key
    colors_dict[key] = color

t_in = {k: truth_dict[k] for k in labels} if point_est else None

multi_corner(s_in, labels, point_est=t_in, colors_dict=colors_dict, labels_dict=labels_dict)